# Stage 5 — controlled XGBoost tuning

Stage 4 supported tuning XGBoost only. Random Forest was not tuned because its untuned candidate had lower bad-risk recall, higher 5:1 cost, and greater overfitting evidence. This notebook orchestrates reusable nested-CV code; the final holdout remains untouched.

## Why nested cross-validation

The persisted Stage 4 folds are outer evaluation folds. Inside each 640-row outer-training partition, a four-fold randomized search samples the same 40 configurations and selects by Average Precision. The selected pipeline is refit only on that outer-training partition before predicting its 160 held-out outer-validation rows. Combining those predictions gives an honest development estimate of the complete tuning procedure.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from creditscope.stage5 import SEARCH_SPACE, generate_stage5_artifacts

project_root = Path.cwd()
display(pd.DataFrame({'hyperparameter': SEARCH_SPACE.keys(), 'values': [str(value) for value in SEARCH_SPACE.values()]}))
run_summary = generate_stage5_artifacts(project_root)
run_summary

## Honest nested OOF performance and unchanged-threshold cost

In [ ]:
display(pd.read_csv(project_root / 'reports/stage5/nested_model_comparison.csv'))
display(pd.read_csv(project_root / 'reports/stage5/outer_fold_metrics.csv'))
display(pd.read_csv(project_root / 'reports/stage5/outer_fold_metric_summary.csv'))
display(pd.read_csv(project_root / 'reports/stage5/tuned_vs_untuned.csv'))

## Outer-fold selections and hyperparameter stability

In [ ]:
display(pd.read_csv(project_root / 'reports/stage5/outer_fold_best_params.csv'))
display(pd.read_csv(project_root / 'reports/stage5/hyperparameter_stability.csv'))
display(pd.read_csv(project_root / 'reports/stage5/prediction_change_summary.csv'))

## Ranking, calibration, overfitting, and probability-change diagnostics

All classifications still use 0.50. Reliability curves are diagnostic only; no probability calibration is fitted.

In [ ]:
for figure in [
    'tuned_vs_untuned_roc.png',
    'tuned_vs_untuned_precision_recall.png',
    'confusion_matrices.png',
    'calibration_comparison.png',
    'train_validation_gap.png',
    'probability_change_distribution.png',
    'hyperparameter_stability.png',
]:
    display(Image(filename=project_root / 'reports/stage5/figures' / figure))

## Final development-wide configuration search

Only after nested evaluation, the same 40-draw search is run across all 800 development rows using the locked five folds. Its best score is selection-biased and is not an unbiased performance estimate. `refit=False` means Stage 5 does not create a final all-development estimator.

In [ ]:
display(Markdown((project_root / 'reports/stage5/stage5_tuning_summary.md').read_text(encoding='utf-8')))

## What remains for Stage 6

Stage 6 must decide whether to retain the tuned configuration or the stronger untuned candidate, and whether threshold analysis, calibration, or other evaluation is authorized. No such work occurs here. The final 200-row holdout still has no predictions or performance metrics.